# Sequence Dataset For Stockout Forecasting

Reusable preprocessing and time-window construction for daily 7-day stockout prediction.

In [1]:
# Colab bootstrap: clone or update the repo, then run from its root.
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/vibhor-5/btp.git"
REPO_DIR = Path("/content/btp")
try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    os.chdir(REPO_DIR)
else:
    os.chdir(Path.cwd())

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
TRAIN_PATH = DATA_DIR / "top15_train.parquet"
TEST_PATH = DATA_DIR / "top15_test.parquet"
print("Project root:", PROJECT_ROOT)
print("Train exists:", TRAIN_PATH.exists(), TRAIN_PATH)
print("Test exists:", TEST_PATH.exists(), TEST_PATH)

try:
    import pyarrow  # noqa: F401
except ImportError:
    !pip -q install pyarrow

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader


Project root: /content/btp
Train exists: True /content/btp/data/top15_train.parquet
Test exists: True /content/btp/data/top15_test.parquet


In [2]:
@dataclass
class Config:
    history_len: int = 28
    forecast_len: int = 7
    batch_size: int = 256
    val_days: int = 15
    seed: int = 42

CFG = Config()

STATIC_CAT_COLS = [
    "city_id", "store_id", "management_group_id", "first_category_id",
    "second_category_id", "third_category_id", "product_id",
]
TEMPORAL_CAT_COLS = ["holiday_flag", "activity_flag"]
CAT_COLS = STATIC_CAT_COLS + TEMPORAL_CAT_COLS

HIST_NUM_COLS = ["discount", "precpt", "avg_temperature", "avg_humidity", "avg_wind_level", "sale_amount", "stockout"]
FUTURE_NUM_COLS = ["discount", "precpt", "avg_temperature", "avg_humidity", "avg_wind_level"]
TARGET_COL = "stockout"

In [3]:
def load_data() -> Tuple[pd.DataFrame, pd.DataFrame]:
    train = pd.read_parquet(TRAIN_PATH)
    test = pd.read_parquet(TEST_PATH)
    for df in (train, test):
        df["dt"] = pd.to_datetime(df["dt"])
        df[TARGET_COL] = (df["stock_hour6_22_cnt"] > 0).astype(np.float32)
    return train, test

def chronological_train_val_dates(train_df: pd.DataFrame, val_days: int) -> Tuple[pd.Timestamp, pd.Timestamp]:
    dates = np.array(sorted(train_df["dt"].unique()))
    if len(dates) <= val_days:
        raise ValueError("Not enough train dates for validation split")
    val_start = pd.Timestamp(dates[-val_days])
    train_end = pd.Timestamp(dates[-val_days - 1])
    return train_end, val_start

train_df, test_df = load_data()
train_end, val_start = chronological_train_val_dates(train_df, CFG.val_days)
print("train rows", train_df.shape, "test rows", test_df.shape)
print("train target end <=", train_end, "validation target start >=", val_start)

train rows (175575, 20) test rows (35115, 20)
train target end <= 2024-05-26 00:00:00 validation target start >= 2024-05-27 00:00:00


In [4]:
class CategoryEncoder:
    def __init__(self, cols: List[str]):
        self.cols = cols
        self.maps: Dict[str, Dict[str, int]] = {}
        self.sizes: Dict[str, int] = {}

    def fit(self, df: pd.DataFrame):
        for col in self.cols:
            values = df[col].fillna("__NA__").astype(str).unique().tolist()
            mapping = {"__UNK__": 0}
            mapping.update({v: i + 1 for i, v in enumerate(sorted(values))})
            self.maps[col] = mapping
            self.sizes[col] = len(mapping)
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        for col in self.cols:
            mapping = self.maps[col]
            out[col] = out[col].fillna("__NA__").astype(str).map(mapping).fillna(0).astype(np.int64)
        return out

def fit_preprocessors(train_df: pd.DataFrame, train_end: pd.Timestamp):
    fit_rows = train_df[train_df["dt"] <= train_end].copy()
    cat_encoder = CategoryEncoder(CAT_COLS).fit(fit_rows)
    scaler = StandardScaler()
    scale_cols = sorted((set(HIST_NUM_COLS + FUTURE_NUM_COLS) - {TARGET_COL}))
    scaler.fit(fit_rows[scale_cols].fillna(0.0))
    return cat_encoder, scaler

def apply_preprocessors(df: pd.DataFrame, cat_encoder: CategoryEncoder, scaler: StandardScaler) -> pd.DataFrame:
    df = df.copy()
    df = cat_encoder.transform(df)
    scale_cols = sorted((set(HIST_NUM_COLS + FUTURE_NUM_COLS) - {TARGET_COL}))
    df[scale_cols] = scaler.transform(df[scale_cols].fillna(0.0))
    return df

cat_encoder, scaler = fit_preprocessors(train_df, train_end)
combined = pd.concat([train_df.assign(source="train"), test_df.assign(source="test")], ignore_index=True)
encoded = apply_preprocessors(combined, cat_encoder, scaler)
print(cat_encoder.sizes)

{'city_id': 19, 'store_id': 869, 'management_group_id': 4, 'first_category_id': 9, 'second_category_id': 10, 'third_category_id': 13, 'product_id': 16, 'holiday_flag': 3, 'activity_flag': 3}


In [5]:
def make_windows(df: pd.DataFrame, cfg: Config, train_end: pd.Timestamp, val_start: pd.Timestamp):
    rows = []
    df = df.sort_values(["store_id", "product_id", "dt"]).reset_index(drop=True)
    for (store_id, product_id), group in df.groupby(["store_id", "product_id"], sort=False):
        group = group.sort_values("dt")
        if len(group) < cfg.history_len + cfg.forecast_len:
            continue
        hist_num = group[HIST_NUM_COLS].to_numpy(np.float32)
        fut_num = group[FUTURE_NUM_COLS].to_numpy(np.float32)
        cats = group[CAT_COLS].to_numpy(np.int64)
        target = group[TARGET_COL].to_numpy(np.float32)
        dates = group["dt"].to_numpy()

        for start in range(0, len(group) - cfg.history_len - cfg.forecast_len + 1):
            hist_end = start + cfg.history_len
            fut_end = hist_end + cfg.forecast_len
            target_start = pd.Timestamp(dates[hist_end])
            target_end = pd.Timestamp(dates[fut_end - 1])
            if target_end <= train_end:
                split = "train"
            elif target_start >= val_start and target_end <= train_df["dt"].max():
                split = "val"
            elif target_start > train_df["dt"].max():
                split = "test"
            else:
                continue

            rows.append({
                "hist_num": hist_num[start:hist_end],
                "hist_cat": cats[start:hist_end],
                "future_num": fut_num[hist_end:fut_end],
                "future_cat": cats[hist_end:fut_end],
                "target": target[hist_end:fut_end],
                "target_start": target_start,
                "target_end": target_end,
                "split": split,
                "store_id": store_id,
                "product_id": product_id,
            })
    return rows

windows = make_windows(encoded, CFG, train_end, val_start)
split_counts = pd.Series([w["split"] for w in windows]).value_counts()
display(split_counts)

,count
train,60866
val,21069
test,21069


In [6]:
class StockoutSequenceDataset(Dataset):
    def __init__(self, windows, split: str):
        self.windows = [w for w in windows if w["split"] == split]

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        return {
            "hist_num": torch.tensor(w["hist_num"], dtype=torch.float32),
            "hist_cat": torch.tensor(w["hist_cat"], dtype=torch.long),
            "future_num": torch.tensor(w["future_num"], dtype=torch.float32),
            "future_cat": torch.tensor(w["future_cat"], dtype=torch.long),
            "target": torch.tensor(w["target"], dtype=torch.float32),
        }

datasets = {split: StockoutSequenceDataset(windows, split) for split in ["train", "val", "test"]}
loaders = {
    "train": DataLoader(datasets["train"], batch_size=CFG.batch_size, shuffle=True, num_workers=0),
    "val": DataLoader(datasets["val"], batch_size=CFG.batch_size, shuffle=False, num_workers=0),
    "test": DataLoader(datasets["test"], batch_size=CFG.batch_size, shuffle=False, num_workers=0),
}

batch = next(iter(loaders["train"]))
for key, value in batch.items():
    print(key, tuple(value.shape), value.dtype)

assert not torch.isnan(batch["hist_num"]).any()
assert not torch.isnan(batch["future_num"]).any()
assert batch["target"].shape[1] == CFG.forecast_len
print("Dataset checks passed")

hist_num (256, 28, 7) torch.float32
hist_cat (256, 28, 9) torch.int64
future_num (256, 7, 5) torch.float32
future_cat (256, 7, 9) torch.int64
target (256, 7) torch.float32
Dataset checks passed
